# basline.ipynb

Qwen2-Audio baseline inference, prompt evaluation, and representation extraction.

Set the audio and metadata paths, choose the experiment section, and run its setup and inference cells.

Use the setup steps in the repository README before running this notebook.


In [ ]:
# ============================================================
# QWEN2-AUDIO BASELINE DEVIATION RUN
# FULL VERSION WITH TASK-AWARE PROMPTS + SELECTED LAYER SAVING
# ============================================================

import os
import re
import json
import random
import librosa
import torch
import pandas as pd
from tqdm import tqdm
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "Qwen/Qwen2-Audio-7B-Instruct"
BASE_DIR = "/content/drive/MyDrive/probing"
#SAVE_DIR = "/content/drive/MyDrive/probing/baseline_outputs_edited"

SAVE_DIR = "/content/drive/MyDrive/probing/full_probe/albe"
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SELECTED_AUDIO_LAYERS = [4, 16, 31]
SELECTED_LM_LAYERS = [8, 20, -1]

# ============================================================
# PROMPTS
# ============================================================

# Delivery
SYSTEM_PROMPT_DELIVERY = """
You are a speech emotion classifier.

Determine whether the speaker sounds neutral or emotional.

Choices:
[neutral, emotional]

Return ONLY valid JSON:
{"label": "<neutral or emotional>", "confidence": <0.0-1.0>}
"""

USER_PROMPT_DELIVERY = "Classify the speech."

# Intensity
def build_intensity_prompt(target_emotion, related_emotion=None):

    if related_emotion:
        choices = f"[neutral, {target_emotion}, {related_emotion}]"
    else:
        choices = f"[neutral, {target_emotion}]"

    return f"""
You are a speech emotion classifier.

Determine the emotion of the speaker.

Choices:
{choices}

Return ONLY valid JSON:
{{"label": "<...>", "confidence": <0.0-1.0>}}
"""

USER_PROMPT_INTENSITY = "Classify the emotion of the speech."



# ============================================================
# MODEL LOAD
# ============================================================

processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype="float32",
    device_map="auto"
)

model.eval()

In [ ]:
print(f"Model dtype: {model.dtype}")

In [ ]:
BASE_DIR = "/content/drive/MyDrive/probing"
#SAVE_DIR = "/content/drive/MyDrive/probing/baseline_outputs_edited"

SAVE_DIR = "/content/drive/MyDrive/probing/full_probe/albe"


In [ ]:
CLUSTERS = [
    ["anger", "annoyance", "disapproval"],
    ["disgust", "disapproval"],
    ["fear", "nervousness"],
    ["joy", "amusement", "excitement", "pride", "optimism", "love", "admiration", "relief", "gratitude", "caring", "desire"],
    ["confusion", "curiosity", "realization", "surprise"],
    ["sadness", "disappointment", "remorse", "grief"],
    ["surprise", "realization"]
]
emotion_to_cluster = {}

for cluster in CLUSTERS:
    for e in cluster:
        emotion_to_cluster[e] = cluster
import random
def sample_target_emotion(emotion_list):

    if isinstance(emotion_list, str):
        # if stored as string like "['a','b']"
        import ast
        emotion_list = ast.literal_eval(emotion_list)


    if not emotion_list:
        raise ValueError("Empty emotion list")

    return random.choice(emotion_list)

def pick_distractor(target):

    cluster = emotion_to_cluster.get(target, None)
    print(cluster)

    # Prefer same-cluster distractor
    if cluster:
        candidates = [e for e in cluster if e != target]
        if candidates:
            return random.choice(candidates)

    # fallback (rare)
    all_emotions = list(emotion_to_cluster.keys())
    candidates = [e for e in all_emotions if e != target]
    return random.choice(candidates)
def build_subtle_prompt(target_emotion):
    target = sample_target_emotion(target_emotion)
    print(target)

    distractor = pick_distractor(target_emotion)
    print(distractor)

    choices = [
        target,
        distractor,
        "neutral"
    ]

    random.shuffle(choices)

    choice_str = "\n".join(
        [f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]
    )


    system_prompt = """
You are a speech emotion classifier.

Choose the most appropriate fine-grained emotion expressed in the speaker’s voice.

You must choose ONLY ONE option.

Return ONLY valid JSON:
{"label": "<one of the choices>", "confidence": <0.0-1.0>}
"""

    user_prompt = f"""
Which of the following best describes the speaker’s emotion?

Choices:
{choice_str}
"""

    return system_prompt, user_prompt

In [ ]:
pick_distractor("['joy','fear']")

In [ ]:
import gc
import torch

def clear_memory():
    """Clear PyTorch cache and force garbage collection"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# ============================================================
# PATH REMAP
# ============================================================

def remap_to_colab_path(old_path):

    if not isinstance(old_path, str):
        raise ValueError(f"Invalid path: {old_path}")

    # normalize Windows → Unix
    old_path = old_path.replace("\\", "/").strip()

    # join with Drive base
    new_path = os.path.join(BASE_DIR, old_path)

    return new_path

# ============================================================
# JSON SAFE PARSER
# ============================================================


def parse_json_safe(text):

    if not isinstance(text, str):
        return None, None

    raw = text.strip()

    # ---- remove markdown fences ----
    raw = re.sub(r"^```json", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"^```", "", raw).strip()
    raw = re.sub(r"```$", "", raw).strip()

    # ---- try strict JSON first ----
    try:
        return _parse_valid_json(raw)
    except:
        pass

    # ---- extract JSON substring ----
    start = raw.find("{")
    end = raw.rfind("}")

    if start != -1 and end != -1:
        json_str = raw[start:end+1]

        # fix common issues
        json_str = re.sub(r"(\w+):", r'"\1":', json_str)   # label: → "label":
        json_str = json_str.replace("'", '"')              # ' → "
        json_str = re.sub(r",\s*}", "}", json_str)

        try:
            return _parse_valid_json(json_str)
        except:
            pass

    # ---- fallback: regex extraction ----
    label = None
    confidence = None

    # label
    label_match = re.search(
        r"(neutral|emotional|happy|sad|angry|fearful|disgust|caring|supportive|curious)",
        raw.lower()
    )

    if label_match:
        label = label_match.group(1)

    # confidence
    conf_match = re.search(r"(0\.\d+|1\.0+)", raw)

    if conf_match:
        confidence = float(conf_match.group(1))

    # ---- debug print ----
    #print("⚠️ Fallback parsing used")
    #print("RAW OUTPUT:", raw)

    return label, confidence



def _parse_valid_json(text):
    label, confidence = json.loads(text)
    return label, confidence

# ============================================================
# AUDIO LOAD (UPDATED FOR AUDIO FLAMINGO)
# ============================================================

def load_audio(wav_path):
    # Audio Flamingo expects raw numpy array
    audio, sr = librosa.load(wav_path, sr=16000)  # AF3 uses 16kHz
    return audio

# ============================================================
# HOOK STORAGE
# ============================================================

activations = {}

def make_hook(name):
    def hook(module, inp, out):
        if isinstance(out, tuple):
            out = out[0]
        activations[name] = out.detach().cpu()
    return hook

hooks = []

# def select_three_regions(num_layers):
#     """
#     Return 3 early, 3 mid, 3 late layer indices
#     """
#     if num_layers < 9:
#         raise ValueError(f"Too few layers ({num_layers}) to select 3x3 regions")

#     early = list(range(0, 3))

#     mid_start = num_layers // 2 - 1
#     mid = list(range(mid_start, mid_start + 3))

#     late = list(range(num_layers - 3, num_layers))

#     return early + mid + late
def select_three_regions(num_layers):
    """
    Return ALL layers (modified from original)
    """
    return list(range(num_layers))
num_lm_layers = len(model.language_model.model.layers)
SELECTED_LM_LAYERS = select_three_regions(num_lm_layers)
# LM hooks (AF3 uses Qwen2 with 28 layers)
for idx in SELECTED_LM_LAYERS:
    h = model.language_model.model.layers[idx].register_forward_hook(
        make_hook(f"lm_{idx}")
    )
    hooks.append(h)

# ============================================================
# POOLING
# ============================================================

def mean_pool(x):
    if x.ndim == 3:
        return x.mean(dim=1).squeeze(0)
    elif x.ndim == 2:
        return x.mean(dim=0)
    else:
        return x.flatten()

# ============================================================
# PROMPT ROUTER (UPDATED FOR AF3 CHAT FORMAT)
# ============================================================
def extract_emotion_from_filename(path):
    filename = os.path.basename(path)

    parts = filename.split("_")
    if len(parts) < 3:
        return None

    code = parts[2]

    mapping = {
        "ANG": "angry",
        "DIS": "disgust",
        "FEA": "fearful",
        "HAP": "happy",
        "SAD": "sad",
        "NEU": "neutral"
    }

    return mapping.get(code, None)

def get_prompt_for_row(row):

    task = row["task"]

    if task == "style":
        return SYSTEM_PROMPT_DELIVERY, USER_PROMPT_DELIVERY

    elif task == "intensity":
        target = extract_emotion_from_filename(row["neutral_path"])
        print(target)
        related = row.get("related_emotion", None)

        system_prompt = build_intensity_prompt(target, related)
        return system_prompt, USER_PROMPT_INTENSITY

    # elif task == "subtle":

    #     target = row.get("emotion", None)

    #     if target is None:
    #         raise ValueError(f"Missing subtle emotion label for row {row['id']}")

    #     return build_subtle_prompt(target)

    else:
        raise ValueError(f"Unknown task type: {task}")


# ============================================================
# MODEL INFERENCE (CONVERTED FOR AUDIO FLAMINGO)
# ============================================================

# ============================================================
# MODEL INFERENCE
# ============================================================

def judge_qwen2_audio(
    wav_path,
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_new_tokens=64
):

    import librosa

    global activations
    activations = {}

    # =========================
    # LOAD AUDIO (CORRECT WAY)
    # =========================
    waveform, _ = librosa.load(
        wav_path,
        sr=processor.feature_extractor.sampling_rate
    )

    # =========================
    # CONVERSATION FORMAT
    # =========================
    conversation = [
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": f"file://{wav_path}"},
                {"type": "text", "text": f"{system_prompt}\n\n{user_prompt}"},
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False
    )

    # =========================
    # PROCESS INPUTS (CRITICAL FIX)
    # =========================
    inputs = processor(
        text=text,
        audio=[waveform],  # ✅ correct key
        sampling_rate=processor.feature_extractor.sampling_rate,  # ✅ required
        return_tensors="pt",
        padding=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # =========================
    # AUDIO FORWARD (EXPLICIT)
    # =========================
    with torch.no_grad():
        audio_out = model.audio_tower(
            input_features=inputs["input_features"],
            input_features_mask=inputs.get("feature_attention_mask", None),
            output_hidden_states=True
        )

    audio_hidden_states = audio_out.hidden_states
    selected_audio_idx = select_three_regions(len(audio_hidden_states))

    pooled_reps = {}

    # ---- audio layers (balanced sampling) ----
    for i in selected_audio_idx:
        pooled_reps[f"audio_{i}"] = audio_hidden_states[i].mean(dim=1).cpu()
    # =========================
    # PROJECTOR
    # =========================
    with torch.no_grad():
        projected = model.multi_modal_projector(audio_hidden_states[-1])

    pooled_reps["projector"] = projected.mean(dim=1).cpu()

    # =========================
    # FORWARD + GENERATE
    # =========================
    with torch.no_grad():

        # Forward pass (for hooks)
        _ = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True
        )

        generate_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    # =========================
    # DECODE OUTPUT
    # =========================
    generate_ids = generate_ids[:, inputs["input_ids"].shape[1]:]

    response = processor.batch_decode(
        generate_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    label, confidence = parse_json_safe(response)

    # =========================
    # POOL ACTIVATIONS
    # =========================

    # ---- LM activations ----
    for k, v in activations.items():
        pooled_reps[k] = v.mean(dim=1).cpu()   # full tensor (if disk allows)

    # =========================
    # CLEANUP
    # =========================
    del generate_ids
    del inputs
    activations.clear()

    return label, confidence, response , pooled_reps
# ============================================================
# SAVE REPRESENTATIONS
# ============================================================

def save_representation_tensor(rep_dict, save_path):
    torch.save(rep_dict, save_path)

# ============================================================
# SAMPLE SUBSET PER TASK
# ============================================================

def sample_subset(paths, fraction):
    n = max(10, int(len(paths) * fraction))
    return random.sample(paths, min(n, len(paths)))

# ============================================================
# MAIN BASELINE RUNNER (UPDATED FUNCTION NAME)
# ============================================================

def run_baseline_deviation_af3(metadata):

    rows = []

    grouped = metadata.groupby("baseline_type")

    for baseline_type, group_df in grouped:

        print(f"\n===== Running baseline group: {baseline_type} =====")

        unique_rows = group_df.drop_duplicates(subset=["neutral_path"])

        temp0_rows = unique_rows.sample(
            n=max(10, int(len(unique_rows) * 0.10)),
            random_state=42
        )

        temp03_rows = unique_rows.sample(
            n=max(10, int(len(unique_rows) * 0.30)),
            random_state=42
        )

        # -----------------------
        # temp=0
        # -----------------------
        for _, row in tqdm(temp0_rows.iterrows(), total=len(temp0_rows)):

            wav_path = remap_to_colab_path(row["neutral_path"])

            if not os.path.exists(wav_path):
                continue

            system_prompt, user_prompt = get_prompt_for_row(row)

            for run_idx in range(2):

                label, conf, raw, reps = judge_audio_flamingo(
                    wav_path,
                    system_prompt,
                    user_prompt,
                    temperature=0.0
                )

                rep_path = os.path.join(
                    SAVE_DIR,
                    f"af3_{baseline_type}_{row['id']}_temp0_run{run_idx+1}.pt"
                )

                save_representation_tensor(reps, rep_path)

                rows.append({
                    "id": row["id"],
                    "audio": wav_path,
                    "baseline_type": baseline_type,
                    "temperature": 0.0,
                    "run": run_idx + 1,
                    "label": label,
                    "confidence": conf,
                    "raw_response": raw,
                    "model": "audio_flamingo_3"
                })
                clear_memory()


        # -----------------------
        # temp=0.3
        # -----------------------
        for _, row in tqdm(temp03_rows.iterrows(), total=len(temp03_rows)):

            wav_path = remap_to_colab_path(row["neutral_path"])

            if not os.path.exists(wav_path):
                continue

            system_prompt, user_prompt = get_prompt_for_row(row)

            for run_idx in range(3):

                label, conf, raw, reps = judge_audio_flamingo(
                    wav_path,
                    system_prompt,
                    user_prompt,
                    temperature=0.3
                )

                rep_path = os.path.join(
                    SAVE_DIR,
                    f"af3_{baseline_type}_{row['id']}_temp03_run{run_idx+1}.pt"
                )

                save_representation_tensor(reps, rep_path)

                rows.append({
                    "id": row["id"],
                    "audio": wav_path,
                    "baseline_type": baseline_type,
                    "temperature": 0.3,
                    "run": run_idx + 1,
                    "label": label,
                    "confidence": conf,
                    "raw_response": raw,
                    "model": "audio_flamingo_3"
                })
                clear_memory()

    df = pd.DataFrame(rows)

    csv_path = os.path.join(SAVE_DIR, "audio_flamingo_baseline_results.csv")
    df.to_csv(csv_path, index=False)

    print(f"\nSaved CSV: {csv_path}")

In [ ]:

# # ============================================================
# # PATH REMAP
# # ============================================================

# def remap_to_colab_path(old_path):

#     if not isinstance(old_path, str):
#         raise ValueError(f"Invalid path: {old_path}")

#     # normalize Windows → Unix
#     old_path = old_path.replace("\\", "/").strip()

#     # join with Drive base
#     new_path = os.path.join(BASE_DIR, old_path)

#     return new_path
# # ============================================================
# # JSON SAFE PARSER
# # ============================================================

# def parse_json_safe(text):

#     if not isinstance(text, str):
#         return None, None

#     raw = text.strip()

#     # ---- remove markdown fences ----
#     raw = re.sub(r"^```json", "", raw, flags=re.IGNORECASE).strip()
#     raw = re.sub(r"^```", "", raw).strip()
#     raw = re.sub(r"```$", "", raw).strip()

#     # ---- try strict JSON first ----
#     try:
#         return _parse_valid_json(raw)
#     except:
#         pass

#     # ---- extract JSON substring ----
#     start = raw.find("{")
#     end = raw.rfind("}")

#     if start != -1 and end != -1:
#         json_str = raw[start:end+1]

#         # fix common issues
#         json_str = re.sub(r"(\w+):", r'"\1":', json_str)   # label: → "label":
#         json_str = json_str.replace("'", '"')              # ' → "
#         json_str = re.sub(r",\s*}", "}", json_str)

#         try:
#             return _parse_valid_json(json_str)
#         except:
#             pass

#     # ---- fallback: regex extraction ----
#     label = None
#     confidence = None

#     # label
#     label_match = re.search(
#         r"(neutral|emotional|happy|sad|angry|fearful|disgust|caring|supportive|curious)",
#         raw.lower()
#     )

#     if label_match:
#         label = label_match.group(1)

#     # confidence
#     conf_match = re.search(r"(0\.\d+|1\.0+)", raw)

#     if conf_match:
#         confidence = float(conf_match.group(1))

#     # ---- debug print ----
#     #print("⚠️ Fallback parsing used")
#     #print("RAW OUTPUT:", raw)

#     return label, confidence


# def _parse_valid_json(text):
#     label, confidence = json.loads(text)

#     return label, confidence

# # ============================================================
# # AUDIO LOAD
# # ============================================================

# def load_audio(wav_path):
#     audio, sr = librosa.load(
#         wav_path,
#         sr=processor.feature_extractor.sampling_rate
#     )
#     return audio

# # ============================================================
# # HOOK STORAGE
# # ============================================================

# activations = {}

# def make_hook(name):
#     def hook(module, inp, out):
#         if isinstance(out, tuple):
#             out = out[0]
#         activations[name] = out.detach().cpu()
#     return hook

# hooks = []



# # def select_three_regions(num_layers):
# #     """
# #     Return 3 early, 3 mid, 3 late layer indices
# #     """
# #     if num_layers < 9:
# #         raise ValueError(f"Too few layers ({num_layers}) to select 3x3 regions")

# #     early = list(range(0, 3))

# #     mid_start = num_layers // 2 - 1
# #     mid = list(range(mid_start, mid_start + 3))

# #     late = list(range(num_layers - 3, num_layers))

# #     return early + mid + late
# def select_three_regions(num_layers):
#     """
#     Return ALL layers (modified from original)
#     """
#     return list(range(num_layers))

# num_lm_layers = len(model.language_model.model.layers)
# SELECTED_LM_LAYERS = select_three_regions(num_lm_layers)
# for idx in SELECTED_LM_LAYERS:
#     h = model.language_model.model.layers[idx].register_forward_hook(
#         make_hook(f"lm_{idx}")
#     )
#     hooks.append(h)

# # ============================================================
# # POOLING
# # ============================================================

# def mean_pool(x):
#     if x.ndim == 3:
#         return x.mean(dim=1).squeeze(0)
#     elif x.ndim == 2:
#         return x.mean(dim=0)
#     else:
#         return x.flatten()

# # ============================================================
# # PROMPT ROUTER
# # ============================================================
# def extract_emotion_from_filename(path):
#     filename = os.path.basename(path)

#     parts = filename.split("_")
#     if len(parts) < 3:
#         return None

#     code = parts[2]

#     mapping = {
#         "ANG": "angry",
#         "DIS": "disgust",
#         "FEA": "fearful",
#         "HAP": "happy",
#         "SAD": "sad",
#         "NEU": "neutral"
#     }

#     return mapping.get(code, None)

# def get_prompt_for_row(row):

#     task = row["task"]

#     if task == "style":
#         return SYSTEM_PROMPT_DELIVERY, USER_PROMPT_DELIVERY

#     elif task == "intensity":
#         target = extract_emotion_from_filename(row["neutral_path"])
#         print(target)
#         related = row.get("related_emotion", None)

#         system_prompt = build_intensity_prompt(target, related)
#         return system_prompt, USER_PROMPT_INTENSITY

#     # elif task == "subtle":

#     #     target = row.get("emotion", None)

#     #     if target is None:
#     #         raise ValueError(f"Missing subtle emotion label for row {row['id']}")

#     #     return build_subtle_prompt(target)

#     else:
#         raise ValueError(f"Unknown task type: {task}")

# # ============================================================
# # MODEL INFERENCE
# # ============================================================

# def judge_qwen2_audio(
#     wav_path,
#     system_prompt,
#     user_prompt,
#     temperature=0.0,
#     max_new_tokens=64
# ):

#     import librosa

#     global activations
#     activations = {}

#     # =========================
#     # LOAD AUDIO (CORRECT WAY)
#     # =========================
#     waveform, _ = librosa.load(
#         wav_path,
#         sr=processor.feature_extractor.sampling_rate
#     )

#     # =========================
#     # CONVERSATION FORMAT
#     # =========================
#     conversation = [
#         {'role': 'system', 'content': 'You are a helpful assistant.'},
#         {
#             "role": "user",
#             "content": [
#                 {"type": "audio", "audio_url": f"file://{wav_path}"},
#                 {"type": "text", "text": f"{system_prompt}\n\n{user_prompt}"},
#             ],
#         }
#     ]

#     text = processor.apply_chat_template(
#         conversation,
#         add_generation_prompt=True,
#         tokenize=False
#     )

#     # =========================
#     # PROCESS INPUTS (CRITICAL FIX)
#     # =========================
#     inputs = processor(
#         text=text,
#         audio=[waveform],  # ✅ correct key
#         sampling_rate=processor.feature_extractor.sampling_rate,  # ✅ required
#         return_tensors="pt",
#         padding=True
#     )

#     inputs = {k: v.to(model.device) for k, v in inputs.items()}

#     # =========================
#     # AUDIO FORWARD (EXPLICIT)
#     # =========================
#     with torch.no_grad():
#         audio_out = model.audio_tower(
#             input_features=inputs["input_features"],
#             input_features_mask=inputs.get("feature_attention_mask", None),
#             output_hidden_states=True
#         )

#     audio_hidden_states = audio_out.hidden_states
#     selected_audio_idx = select_three_regions(len(audio_hidden_states))

#     pooled_reps = {}

#     # ---- audio layers (balanced sampling) ----
#     for i in selected_audio_idx:
#         pooled_reps[f"audio_{i}"] = audio_hidden_states[i].mean(dim=1).cpu()
#     # =========================
#     # PROJECTOR
#     # =========================
#     with torch.no_grad():
#         projected = model.multi_modal_projector(audio_hidden_states[-1])

#     pooled_reps["projector"] = projected.mean(dim=1).cpu()

#     # =========================
#     # FORWARD + GENERATE
#     # =========================
#     with torch.no_grad():

#         # Forward pass (for hooks)
#         _ = model(
#             **inputs,
#             output_hidden_states=True,
#             return_dict=True
#         )

#         generate_ids = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=(temperature > 0),
#             temperature=temperature,
#             eos_token_id=processor.tokenizer.eos_token_id
#         )

#     # =========================
#     # DECODE OUTPUT
#     # =========================
#     generate_ids = generate_ids[:, inputs["input_ids"].shape[1]:]

#     response = processor.batch_decode(
#         generate_ids,
#         skip_special_tokens=True,
#         clean_up_tokenization_spaces=False
#     )[0]

#     label, confidence = parse_json_safe(response)

#     # =========================
#     # POOL ACTIVATIONS
#     # =========================

#     # ---- LM activations ----
#     for k, v in activations.items():
#         pooled_reps[k] = v.mean(dim=1).cpu()   # full tensor (if disk allows)

#     # =========================
#     # CLEANUP
#     # =========================

#     return label, confidence, response, pooled_reps

# # ============================================================
# # SAVE REPRESENTATIONS
# # ============================================================

# def save_representation_tensor(rep_dict, save_path):
#     torch.save(rep_dict, save_path)

# # ============================================================
# # SAMPLE SUBSET PER TASK
# # ============================================================

# def sample_subset(paths, fraction):
#     n = max(10, int(len(paths) * fraction))
#     return random.sample(paths, min(n, len(paths)))

# # ============================================================
# # MAIN BASELINE RUNNER
# # ============================================================

# def run_baseline_deviation(metadata):

#     rows = []

#     grouped = metadata.groupby("baseline_type")

#     for baseline_type, group_df in grouped:

#         print(f"\n===== Running baseline group: {baseline_type} =====")

#         unique_rows = group_df.drop_duplicates(subset=["neutral_path"])

#         temp0_rows = unique_rows.sample(
#             n=max(10, int(len(unique_rows) * 0.10)),
#             random_state=42
#         )

#         temp03_rows = unique_rows.sample(
#             n=max(10, int(len(unique_rows) * 0.30)),
#             random_state=42
#         )

#         # -----------------------
#         # temp=0
#         # -----------------------
#         for _, row in tqdm(temp0_rows.iterrows(), total=len(temp0_rows)):

#             wav_path = remap_to_colab_path(row["neutral_path"])

#             if not os.path.exists(wav_path):
#                 print('change path')
#                 continue

#             system_prompt, user_prompt = get_prompt_for_row(row)

#             for run_idx in range(2):

#                 label, conf, raw, reps = judge_qwen2_audio(
#                     wav_path,
#                     system_prompt,
#                     user_prompt,
#                     temperature=0.0
#                 )

#                 rep_path = os.path.join(
#                     SAVE_DIR,
#                     f"{baseline_type}_{row['id']}_temp0_run{run_idx+1}.pt"
#                 )

#                 save_representation_tensor(reps, rep_path)

#                 rows.append({
#                     "id": row["id"],
#                     "audio": wav_path,
#                     "baseline_type": baseline_type,
#                     "temperature": 0.0,
#                     "run": run_idx + 1,
#                     "label": label,
#                     "confidence": conf,
#                     "raw_response": raw
#                 })

#         # -----------------------
#         # temp=0.3
#         # -----------------------
#         for _, row in tqdm(temp03_rows.iterrows(), total=len(temp03_rows)):

#             wav_path = remap_to_colab_path(row["neutral_path"])

#             if not os.path.exists(wav_path):
#                 continue

#             system_prompt, user_prompt = get_prompt_for_row(row)

#             for run_idx in range(3):

#                 label, conf, raw, reps = judge_qwen2_audio(
#                     wav_path,
#                     system_prompt,
#                     user_prompt,
#                     temperature=0.3
#                 )

#                 rep_path = os.path.join(
#                     SAVE_DIR,
#                     f"{baseline_type}_{row['id']}_temp03_run{run_idx+1}.pt"
#                 )

#                 save_representation_tensor(reps, rep_path)

#                 rows.append({
#                     "id": row["id"],
#                     "audio": wav_path,
#                     "baseline_type": baseline_type,
#                     "temperature": 0.3,
#                     "run": run_idx + 1,
#                     "label": label,
#                     "confidence": conf,
#                     "raw_response": raw
#                 })

#     df = pd.DataFrame(rows)

#     csv_path = os.path.join(SAVE_DIR, "baseline_deviation_results.csv")
#     df.to_csv(csv_path, index=False)

#     print(f"\nSaved CSV: {csv_path}")



In [ ]:
# ============================================================
# LOAD METADATA + RUN
# ============================================================

metadata = pd.read_csv("/content/drive/MyDrive/probing/subset_data.csv")
metadata = metadata[metadata["baseline_type"] == 'subtle']

run_baseline_deviation(metadata)

# ============================================================
# REMOVE HOOKS AFTER DONE
# ============================================================

for h in hooks:
    h.remove()

In [ ]:
def run_neutral_pass(metadata):

    results = {}
    rows = []

    unique_neutral = metadata["neutral_path"].drop_duplicates()

    print(f"\n===== Running NEUTRAL pass ({len(unique_neutral)} files) =====")

    for path in tqdm(unique_neutral):

        wav_path = remap_to_colab_path(path)

        if not os.path.exists(wav_path):
            print(f"⚠️ Missing: {wav_path}")
            continue

        row = metadata[metadata["neutral_path"] == path].iloc[0]
        system_prompt, user_prompt = get_prompt_for_row(row)

        try:
            label, conf, raw, reps = judge_qwen2_audio(
                wav_path,
                system_prompt,
                user_prompt,
                temperature=0.0
            )
        except Exception as e:
            print(f"❌ Error: {wav_path} | {e}")
            continue

        rep_path = os.path.join(
            SAVE_DIR,
            f"neutral_{row['id']}.pt"
        )
        save_representation_tensor(reps, rep_path)

        results[path] = {
            "label": label,
            "confidence": conf,
            "raw": raw,
            "rep_path": rep_path
        }

        rows.append({
            "neutral_path": path,
            "label": label,
            "confidence": conf,
            "raw_response": raw,
            "rep_path": rep_path
        })

    # ✅ save CSV
    df = pd.DataFrame(rows)
    csv_path = os.path.join(SAVE_DIR, "neutral_results.csv")
    df.to_csv(csv_path, index=False)

    print(f"Saved neutral CSV: {csv_path}")

    return results

In [ ]:
def run_manipulated_pass(metadata):

    rows = []

    print(f"\n===== Running MANIPULATED pass ({len(metadata)} samples) =====")

    for _, row in tqdm(metadata.iterrows(), total=len(metadata)):

        wav_path = remap_to_colab_path(row["manipulated_path"])

        if not os.path.exists(wav_path):
            print(f"⚠️ Missing: {wav_path}")
            continue

        system_prompt, user_prompt = get_prompt_for_row(row)

        try:
            label, conf, raw, reps = judge_qwen2_audio(
                wav_path,
                system_prompt,
                user_prompt,
                temperature=0.0
            )
        except Exception as e:
            print(f"❌ Error: {wav_path} | {e}")
            continue

        rep_path = os.path.join(
            SAVE_DIR,'reps',
            f"manip_{row['id']}_{row['intensity']}_{row['condition']}_{row['baseline_type']}.pt"
        )
        save_representation_tensor(reps, rep_path)

        rows.append({
            "id": row["id"],
            "task": row["task"],
            "neutral_path": row["neutral_path"],
            "manipulated_path": row["manipulated_path"],
            "label": label,
            "confidence": conf,
            "raw_response": raw,
            "rep_path": rep_path
        })

    df = pd.DataFrame(rows)

    csv_path = os.path.join(SAVE_DIR, "manipulated_results.csv")
    df.to_csv(csv_path, index=False)

    print(f"Saved manipulated CSV: {csv_path}")

    return df

In [ ]:
def merge_and_save(manip_df, neutral_results):

    merged_rows = []

    for _, row in manip_df.iterrows():

        neutral_info = neutral_results.get(row["neutral_path"], None)

        if neutral_info is None:
            continue

        merged_rows.append({
            "id": row["id"],
            "task": row["task"],

            "neutral_path": row["neutral_path"],
            "manipulated_path": row["manipulated_path"],

            # manipulated
            "label": row["label"],
            "confidence": row["confidence"],
            "raw_response": row["raw_response"],
            "rep_path": row["rep_path"],

            # neutral
            "neutral_label": neutral_info["label"],
            "neutral_confidence": neutral_info["confidence"],
            "neutral_raw": neutral_info["raw"],
            "neutral_rep_path": neutral_info["rep_path"]
        })

    df = pd.DataFrame(merged_rows)

    csv_path = os.path.join(SAVE_DIR, "merged_results.csv")
    df.to_csv(csv_path, index=False)

    print(f"Saved merged CSV: {csv_path}")

    return df

In [ ]:
# ============================================================
# LOAD METADATA + RUN
# ============================================================

#metadata = pd.read_csv("/content/drive/MyDrive/probing/subset_data.csv")
#metadata = metadata[metadata["baseline_type"] == 'subtle']

#manip_df = run_manipulated_pass(metadata)
#merged_df = merge_and_save(manip_df, neutral_results)
metadata = pd.read_csv("/content/drive/MyDrive/probing/subset_data.csv")
#neutral_path = os.path.join(SAVE_DIR, "neutral_results.csv")
metadata = metadata[metadata["baseline_type"] != 'subtle']
neutral_results = run_neutral_pass(metadata)
#neutral_results = pd.read_csv(neutral_path)
manip_df = run_manipulated_pass(metadata)
merged_df = merge_and_save(manip_df, neutral_results)

# ============================================================
# REMOVE HOOKS AFTER DONE
# ============================================================

for h in hooks:
    h.remove()

In [ ]:
import torch

def inspect_pt(path, verbose=False):

    data = torch.load(path, map_location="cpu")

    print(f"\n📦 File: {path}")

    # If wrapped dict (recommended format)
    if isinstance(data, dict) and "reps" in data:
        reps = data["reps"]
        print("Contains metadata:")
        for k in data:
            if k != "reps":
                print(f"  {k}: {data[k]}")
    else:
        reps = data

    print("\n🔑 Keys:")
    for k in reps.keys():
        print(f"  - {k}")

    print("\n📊 Tensor Info:")
    for k, v in reps.items():

        if not isinstance(v, torch.Tensor):
            print(f"{k}: ❌ Not a tensor ({type(v)})")
            continue

        print(f"{k}: shape={tuple(v.shape)}, dtype={v.dtype}")

        if verbose:
            print(f"   mean={v.mean().item():.4f}, std={v.std().item():.4f}")

            # print small tensors fully
            if v.numel() < 20:
                print(f"   values={v}")

    print("\n✅ Done.\n")

In [ ]:
inspect_pt("/content/drive/MyDrive/probing/full_probe/albe/reps/manip_1056_IWW_NEU_XX_nan_sigh_cue.pt")

In [ ]:
SYSTEM_PROMPT_PROBE = "You are a helpful assistant."
USER_PROMPT_PROBE = "What is the emotion in this speech?"

In [ ]:
def extract_representation_flamingo(wav_path):

    label, conf, raw, reps = judge_qwen2_audio(
        wav_path,
        SYSTEM_PROMPT_PROBE,
        USER_PROMPT_PROBE,
        temperature=0.0
    )

    return raw,reps

In [ ]:
import os
import pandas as pd
import torch
from tqdm import tqdm

# =========================
# CONFIG
# =========================
METADATA_CSV = "/content/drive/MyDrive/probe/metadata.csv"
SAVE_DIR = "/content/drive/MyDrive/probe/probe_representations_2"
INDEX_CSV = "/content/drive/MyDrive/probe/probe_representations_2/probe_rep_index.csv"

BEHAVIOR_CSV = "/content/drive/MyDrive/probe/probe_representations_2/probe_behavior_output.csv"
behavior_rows = []
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# LOAD METADATA
# =========================
df = pd.read_csv(METADATA_CSV)

rows = []

print(f"Total samples: {len(df)}")

# =========================
# LOOP
# =========================
for _, row in tqdm(df.iterrows(), total=len(df)):

    BASE_DIR = "/content/drive/MyDrive/probe/probe_additional_1000"

    wav_path = os.path.join(BASE_DIR, row["audio_path"])

    if not os.path.exists(wav_path):
        print(f"⚠️ Missing: {wav_path}")
        continue

    try:
        raw,reps = extract_representation_flamingo(wav_path)
    except Exception as e:
        print(f"❌ Error: {wav_path} | {e}")
        continue

    # =========================
    # SAVE REPRESENTATION
    # =========================
    rep_path = os.path.join(SAVE_DIR, f"{row['id']}.pt")

    torch.save(reps, rep_path)

    # =========================
    # STORE INDEX
    # =========================
    rows.append({
        "id": row["id"],
        "audio_path": wav_path,
        "rep_path": rep_path,
        "emotion": row["emotion"],
        "actor": row["actor"],
        "content": row["content"],
        "intensity": row["intensity"],
        "llm_output": raw,
    })

# =========================
# SAVE INDEX CSV
# =========================
index_df = pd.DataFrame(rows)
index_df.to_csv(INDEX_CSV, index=False)

print(f"\nSaved representations to: {SAVE_DIR}")
print(f"Saved index CSV: {INDEX_CSV}")

In [ ]:
inspect_pt("/content/drive/MyDrive/probe/probe_representations_2/999.pt")

In [ ]:
import os
import pandas as pd
import json
from tqdm import tqdm

BASE_DIR = "/content/drive/MyDrive/subprobe"

def resolve_path(p):
    p = str(p).replace("\\", "/").strip()

    # already absolute (just in case)
    if p.startswith("/content/"):
        return p

    return os.path.join(BASE_DIR, p)
CSV_PATH = os.path.join(BASE_DIR, "merged_subtle_emotion.csv")

SAVE_DIR = os.path.join(BASE_DIR, "task3_full_run_qwen/layers")
os.makedirs(SAVE_DIR, exist_ok=True)


# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(CSV_PATH)

# Sample 500 random rows
df = df.sample(n=500, random_state=42)  # random_state for reproducibility

print("Total samples:", len(df))


# ============================================================
# PROMPT
# ============================================================
SYSTEM_PROMPT = "You are a helpful assistant."
USER_PROMPT = "What is the emotion in this speech?"


# ============================================================
# RUN
# ============================================================
rows = []

for idx, row in tqdm(df.iterrows(), total=len(df)):

    wav_path = resolve_path(row["audio_path"])

    if not os.path.exists(wav_path):
        print('no path wo')
        continue

    # parse labels
    labels = json.loads(row["labels"])
    counts = json.loads(row["label_counts"])

    try:
        label, conf, raw, reps = judge_qwen2_audio(
            wav_path,
            SYSTEM_PROMPT,
            USER_PROMPT,
            temperature=0.0
        )

        # save representation
        rep_path = os.path.join(SAVE_DIR, f"{idx}.pt")
        torch.save(reps, rep_path)

        rows.append({
            "id": idx,
            "audio_path": wav_path,
            "language": row["language"],
            "labels": row["labels"],
            "label_counts": row["label_counts"],
            "pred_label": label,
            "confidence": conf,
            "raw_output": raw,
            "rep_path": rep_path
        })

    except Exception as e:
        print(f"Error at {wav_path}: {e}")
        continue


# ============================================================
# SAVE CSV
# ============================================================
out_df = pd.DataFrame(rows)

csv_out = os.path.join(SAVE_DIR, "task3_results.csv")
out_df.to_csv(csv_out, index=False)

print("Saved:", csv_out)

In [ ]:
inspect_pt("/content/drive/MyDrive/subprobe/task3_full_run_qwen/layers/4999.pt")

In [ ]:
import os
import pandas as pd
import json
import random
from tqdm import tqdm
import torch
import gc

# ============================================================
# PATHS
# ============================================================
BASE_DIR = "/content/drive/MyDrive/subprobe"

CSV_PATH = os.path.join(BASE_DIR, "merged_subtle_emotion.csv")
DEFAULT_RESULTS_PATH = os.path.join(BASE_DIR, "task3_full_run_qwen/task3_results.csv")

SAVE_DIR = os.path.join(BASE_DIR, "task3_prompt_subset_v3_qwen")
os.makedirs(SAVE_DIR, exist_ok=True)

CHECKPOINT = os.path.join(SAVE_DIR, "results_partial.csv")
FINAL_OUTPUT = os.path.join(SAVE_DIR, "final_merged_results.csv")


# ============================================================
# PATH RESOLVER
# ============================================================
def resolve_path(p):
    p = str(p).replace("\\", "/").strip()
    if p.startswith("/content/"):
        return p
    return os.path.join(BASE_DIR, p)


# ============================================================
# CLEANUP
# ============================================================
def cleanup():
    gc.collect()
    torch.cuda.empty_cache()


def contains_unsure(labels):
    if isinstance(labels, str):
        labels = json.loads(labels)
    return "unsure" in labels


# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(CSV_PATH)
# drop rows with "unsure"
df_clean = df[~df["labels"].apply(contains_unsure)].copy()


print(f"After filtering: {len(df_clean)} samples")

subset_size = 1000

df_subset = df_clean.sample(
    n=subset_size,
    random_state=42
).reset_index(drop=True)

# save it
SUBSET_PATH = os.path.join(BASE_DIR, "fixed_subset_1000.csv")
df_subset.to_csv(SUBSET_PATH, index=False)



# ============================================================
# CHECKPOINT RESUME
# ============================================================
if os.path.exists(CHECKPOINT):
    existing = pd.read_csv(CHECKPOINT)
    done_keys = set(zip(existing["id"], existing["prompt_type"]))
else:
    existing = pd.DataFrame()
    done_keys = set()

rows = []


# ============================================================
# EMOTION CLUSTERS
# ============================================================
CLUSTERS = [
    ["anger", "annoyance", "disapproval"],
    ["disgust", "disapproval"],
    ["fear", "nervousness"],
    ["joy", "amusement", "excitement", "pride", "optimism", "love", "admiration", "relief", "gratitude", "caring", "desire"],
    ["confusion", "curiosity", "realization", "surprise"],
    ["sadness", "disappointment", "remorse", "grief"],
    ["surprise", "realization"],
]

emotion_to_cluster = {}
for cluster in CLUSTERS:
    for e in cluster:
        emotion_to_cluster[e] = cluster


# ============================================================
# HELPERS
# ============================================================
def sample_target_emotion(emotion_list):
    if isinstance(emotion_list, str):
        emotion_list = json.loads(emotion_list)
    return random.choice(emotion_list)
import json

def is_neutral(labels):
    # handle string case
    if isinstance(labels, str):
        labels = json.loads(labels)

    return len(labels) == 1 and labels[0] == "neutral"

def pick_distractor(target, gt_labels, mode="same_cluster"):
    gt_set = set(gt_labels)
    all_emotions = list(emotion_to_cluster.keys())
    target_cluster = emotion_to_cluster.get(target, [])

    if mode == "same_cluster":
        candidates = [
            e for e in target_cluster
            if e != target and e not in gt_set
        ]
    else:
        candidates = [
            e for e in all_emotions
            if e not in target_cluster and e not in gt_set
        ]

    if not candidates:
        candidates = [
            e for e in all_emotions
            if e != target and e not in gt_set
        ]

    return random.choice(candidates)

import json



def build_subtle_prompt(labels, mode=None):

    target = sample_target_emotion(labels)

    if is_neutral(row["labels"]):
      print("its neutral")

    if mode is None:
        return (
            "You are a helpful assistant.",
            "What subtle emotion is expressed in this speech?",
            target, None, None
        )

    distractor = pick_distractor(target, labels, mode)

    while distractor == target or distractor in labels:
        distractor = pick_distractor(target, labels, mode)

    choices = [target, distractor, "neutral"]
    random.shuffle(choices)

    choice_str = "\n".join(
        [f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]
    )


    system_prompt = """
You are a speech emotion classifier.

Choose the most appropriate fine-grained emotion expressed in the speaker’s voice.

You must choose ONLY ONE option.

Return ONLY valid JSON:
{"label": "<one of the choices>", "confidence": <0.0-1.0>}
"""

    user_prompt = f"""
Which of the following best describes the speaker’s emotion?

Choices:
{choice_str}
"""

    return system_prompt, user_prompt, target, distractor, choices


# ============================================================
# RUN MODEL
# ============================================================
PROMPT_TYPES = ["subtle", "same_cluster", "cross_cluster"]

for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset)):

    wav_path = resolve_path(row["audio_path"])

    if not os.path.exists(wav_path):
        continue

    labels = json.loads(row["labels"])


    for prompt_type in PROMPT_TYPES:

        key = (idx, prompt_type)
        if key in done_keys:
            continue

        try:
            if prompt_type == "subtle":
                system_prompt, user_prompt, target, _, _ = build_subtle_prompt(labels, None)

            elif prompt_type == "same_cluster":
                system_prompt, user_prompt, target, distractor, choices = \
                    build_subtle_prompt(labels, "same_cluster")

            elif prompt_type == "cross_cluster":
                system_prompt, user_prompt, target, distractor, choices = \
                    build_subtle_prompt(labels, "cross_cluster")

            label, conf, raw, reps = judge_qwen2_audio(
                wav_path,
                system_prompt,
                user_prompt,
                temperature=0.0
            )
            rep_dir = os.path.join(SAVE_DIR, "reps", prompt_type)
            os.makedirs(rep_dir, exist_ok=True)

            rep_path = os.path.join(rep_dir, f"{idx}.pt")

            rows.append({
                "id": idx,
                "audio_path": wav_path,
                "labels": row["labels"],
                "language": row["language"],
                "prompt_type": prompt_type,
                "target": target,
                "pred": label,
                "confidence": conf,
                "raw": raw
            })

        except RuntimeError as e:
            if "out of memory" in str(e):
                print("OOM skip")
                cleanup()
                continue
            else:
                print(e)
                continue

        finally:
            cleanup()

    # checkpoint save
    if len(rows) >= 50:
        temp = pd.DataFrame(rows)
        combined = pd.concat([existing, temp], ignore_index=True)
        combined.to_csv(CHECKPOINT, index=False)
        existing = combined
        rows = []


# final save partial
partial_df = pd.concat([existing, pd.DataFrame(rows)], ignore_index=True)
partial_df.to_csv(CHECKPOINT, index=False)


# ============================================================
# MERGE WITH DEFAULT RUN
# ============================================================

print("Merging with default results...")

default_df = pd.read_csv(DEFAULT_RESULTS_PATH)

# normalize paths
default_df["audio_path"] = default_df["audio_path"].apply(resolve_path)
partial_df["audio_path"] = partial_df["audio_path"].apply(resolve_path)

# filter default to subset
subset_paths = set(partial_df["audio_path"])
default_subset = default_df[
    default_df["audio_path"].isin(subset_paths)
].copy()

default_subset = default_subset.rename(columns={
    "pred_label": "default_pred",
    "confidence": "default_conf"
})


# pivot new prompts
pivot_df = partial_df.pivot_table(
    index="audio_path",
    columns="prompt_type",
    values="pred",
    aggfunc="first"
).reset_index()


# merge
final_df = pivot_df.merge(
    default_subset[["audio_path", "default_pred"]],
    on="audio_path",
    how="left"
)

final_df.to_csv(FINAL_OUTPUT, index=False)

print("Final saved:", FINAL_OUTPUT)

In [ ]:
import os
import re
import gc
import json
import random
import librosa
import torch
import pandas as pd
from tqdm import tqdm

# ============================================================
# PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/subprobe"

TEXT_BASELINE_PATH = os.path.join(
    BASE_DIR,
    "fixed_subset_1000_with_text.csv"
)

SAVE_DIR = os.path.join(
    BASE_DIR,
    "task3_text_baseline_qwen"
)

os.makedirs(SAVE_DIR, exist_ok=True)

OUTPUT_CSV = os.path.join(
    SAVE_DIR,
    "text_only_same_cluster_results.csv"
)

CHECKPOINT = os.path.join(
    SAVE_DIR,
    "partial_results.csv"
)

# ============================================================
# CLEANUP
# ============================================================

def clear_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# ============================================================
# JSON PARSER
# ============================================================

def _parse_valid_json(text):

    data = json.loads(text)

    label = data.get("label", None)
    confidence = data.get("confidence", None)

    return label, confidence


def parse_json_safe(text):

    if not isinstance(text, str):
        return None, None

    raw = text.strip()

    # remove markdown fences
    raw = re.sub(r"^```json", "", raw, flags=re.IGNORECASE).strip()
    raw = re.sub(r"^```", "", raw).strip()
    raw = re.sub(r"```$", "", raw).strip()

    # strict parse
    try:
        return _parse_valid_json(raw)
    except:
        pass

    # extract json substring
    start = raw.find("{")
    end = raw.rfind("}")

    if start != -1 and end != -1:

        json_str = raw[start:end+1]

        json_str = re.sub(r"(\w+):", r'"\1":', json_str)
        json_str = json_str.replace("'", '"')
        json_str = re.sub(r",\s*}", "}", json_str)

        try:
            return _parse_valid_json(json_str)
        except:
            pass

    # fallback regex
    label = None
    confidence = None

    label_match = re.search(
        r"(anger|annoyance|disapproval|disgust|fear|nervousness|joy|amusement|excitement|pride|optimism|love|admiration|relief|gratitude|caring|desire|confusion|curiosity|realization|surprise|sadness|disappointment|remorse|grief|neutral)",
        raw.lower()
    )

    if label_match:
        label = label_match.group(1)

    conf_match = re.search(r"(0\.\d+|1\.0+)", raw)

    if conf_match:
        confidence = float(conf_match.group(1))

    return label, confidence

# ============================================================
# HOOK STORAGE
# ============================================================

activations = {}

def make_hook(name):

    def hook(module, inp, out):

        if isinstance(out, tuple):
            out = out[0]

        activations[name] = out.detach().cpu()

    return hook


hooks = []

# ============================================================
# SELECT LAYERS
# ============================================================

def select_three_regions(num_layers):
    return list(range(num_layers))


num_lm_layers = len(model.language_model.model.layers)

SELECTED_LM_LAYERS = select_three_regions(num_lm_layers)

for idx in SELECTED_LM_LAYERS:

    h = model.language_model.model.layers[idx].register_forward_hook(
        make_hook(f"lm_{idx}")
    )

    hooks.append(h)

# ============================================================
# EMOTION CLUSTERS
# ============================================================

CLUSTERS = [
    ["anger", "annoyance", "disapproval"],
    ["disgust", "disapproval"],
    ["fear", "nervousness"],
    ["joy", "amusement", "excitement", "pride", "optimism",
     "love", "admiration", "relief", "gratitude",
     "caring", "desire"],
    ["confusion", "curiosity", "realization", "surprise"],
    ["sadness", "disappointment", "remorse", "grief"],
    ["surprise", "realization"],
]

emotion_to_cluster = {}

for cluster in CLUSTERS:
    for e in cluster:
        emotion_to_cluster[e] = cluster

# ============================================================
# HELPERS
# ============================================================

def sample_target_emotion(emotion_list):

    if isinstance(emotion_list, str):
        emotion_list = json.loads(emotion_list)

    return random.choice(emotion_list)


def pick_distractor(target, gt_labels, mode="same_cluster"):

    gt_set = set(gt_labels)

    all_emotions = list(emotion_to_cluster.keys())

    target_cluster = emotion_to_cluster.get(target, [])

    if mode == "same_cluster":

        candidates = [
            e for e in target_cluster
            if e != target and e not in gt_set
        ]

    else:

        candidates = [
            e for e in all_emotions
            if e not in target_cluster and e not in gt_set
        ]

    if not candidates:

        candidates = [
            e for e in all_emotions
            if e != target and e not in gt_set
        ]

    return random.choice(candidates)

# ============================================================
# PROMPT BUILDER
# ============================================================

def build_same_cluster_prompt(labels):

    if isinstance(labels, str):
        labels = json.loads(labels)

    target = sample_target_emotion(labels)

    distractor = pick_distractor(
        target,
        labels,
        mode="same_cluster"
    )

    while distractor == target or distractor in labels:

        distractor = pick_distractor(
            target,
            labels,
            mode="same_cluster"
        )

    choices = [target, distractor, "neutral"]

    random.shuffle(choices)

    choice_str = "\n".join(
        [f"{chr(65+i)}. {c}" for i, c in enumerate(choices)]
    )

    system_prompt = """
You are an emotion classification assistant.

Choose the most appropriate fine-grained emotion expressed in the transcript.

You must choose ONLY ONE option.

Return ONLY valid JSON:

{"label": "<one of the choices>", "confidence": <0.0-1.0>}
"""

    user_prompt = f"""
Which of the following best describes the speaker's emotion?

Choices:
{choice_str}
"""

    return system_prompt, user_prompt, target, distractor, choices

# ============================================================
# TEXT-ONLY INFERENCE
# ============================================================

def judge_qwen2_text(
    transcript,
    system_prompt,
    user_prompt,
    temperature=0.0,
    max_new_tokens=64
):

    global activations
    activations = {}

    full_user_prompt = f"""
Transcript:
"{transcript}"

{user_prompt}
"""

    conversation = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": full_user_prompt
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False
    )

    inputs = processor(
        text=text,
        return_tensors="pt",
        padding=True
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    pooled_reps = {}

    with torch.no_grad():

        _ = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True
        )

        generate_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature,
            eos_token_id=processor.tokenizer.eos_token_id
        )

    generate_ids = generate_ids[:, inputs["input_ids"].shape[1]:]

    response = processor.batch_decode(
        generate_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    label, confidence = parse_json_safe(response)

    # save LM reps
    for k, v in activations.items():
        pooled_reps[k] = v.mean(dim=1).cpu()

    del generate_ids
    del inputs

    activations.clear()

    return label, confidence, response, pooled_reps

# ============================================================
# SAVE REPS
# ============================================================

def save_representation_tensor(rep_dict, save_path):
    torch.save(rep_dict, save_path)

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(TEXT_BASELINE_PATH)

# ============================================================
# RESUME CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT):

    existing = pd.read_csv(CHECKPOINT)

    done_ids = set(existing["id"])

else:

    existing = pd.DataFrame()

    done_ids = set()

rows = []

# ============================================================
# MAIN LOOP
# ============================================================

for idx, row in tqdm(df.iterrows(), total=len(df)):

    if idx in done_ids:
        continue

    try:

        transcript = row["Utterance"]

        if pd.isna(transcript):
            continue

        labels = json.loads(row["labels"])

        (
            system_prompt,
            user_prompt,
            target,
            distractor,
            choices
        ) = build_same_cluster_prompt(labels)

        label, conf, raw, reps = judge_qwen2_text(
            transcript,
            system_prompt,
            user_prompt,
            temperature=0.0
        )

        # save reps
        rep_dir = os.path.join(SAVE_DIR, "reps")
        os.makedirs(rep_dir, exist_ok=True)

        rep_path = os.path.join(rep_dir, f"{idx}.pt")

        save_representation_tensor(reps, rep_path)

        rows.append({
            "id": idx,
            "audio_path": row["audio_path"],
            "utterance": transcript,
            "labels": row["labels"],
            "target": target,
            "distractor": distractor,
            "choices": json.dumps(choices),
            "pred": label,
            "confidence": conf,
            "raw": raw
        })

    except RuntimeError as e:

        if "out of memory" in str(e).lower():

            print(f"OOM at sample {idx}")

            clear_memory()

            continue

        else:

            print(e)

            continue

    except Exception as e:

        print(f"ERROR at sample {idx}: {e}")

        continue

    finally:

        clear_memory()

    # checkpoint save
    if len(rows) >= 50:

        temp_df = pd.DataFrame(rows)

        combined = pd.concat(
            [existing, temp_df],
            ignore_index=True
        )

        combined.to_csv(CHECKPOINT, index=False)

        existing = combined

        rows = []

# ============================================================
# FINAL SAVE
# ============================================================

final_df = pd.concat(
    [existing, pd.DataFrame(rows)],
    ignore_index=True
)

final_df.to_csv(OUTPUT_CSV, index=False)

print("\nDONE")
print("Saved:", OUTPUT_CSV)

In [ ]:
import os
import pandas as pd
import json
import random
from tqdm import tqdm
import torch
import gc

# ============================================================
# PATHS
# ============================================================
BASE_DIR = "/content/drive/MyDrive/subprobe"

CSV_PATH = os.path.join(BASE_DIR, "merged_subtle_emotion.csv")

DEFAULT_RESULTS_PATH = os.path.join(
    BASE_DIR,
    "task3_full_run_qwen/task3_results.csv"
)

# ------------------------------------------------------------
# NEW MULTIMODAL SAVE DIR
# ------------------------------------------------------------
SAVE_DIR = os.path.join(
    BASE_DIR,
    "task3_prompt_subset_v3_qwen_multimodal"
)

os.makedirs(SAVE_DIR, exist_ok=True)

CHECKPOINT = os.path.join(
    SAVE_DIR,
    "results_partial.csv"
)

FINAL_OUTPUT = os.path.join(
    SAVE_DIR,
    "final_merged_results.csv"
)

# ============================================================
# PATH RESOLVER
# ============================================================
def resolve_path(p):

    p = str(p).replace("\\", "/").strip()

    if p.startswith("/content/"):
        return p

    return os.path.join(BASE_DIR, p)

# ============================================================
# CLEANUP
# ============================================================
def cleanup():

    gc.collect()

    torch.cuda.empty_cache()

def contains_unsure(labels):

    if isinstance(labels, str):
        labels = json.loads(labels)

    return "unsure" in labels

# ============================================================
# LOAD DATA
# ============================================================

# ------------------------------------------------------------
# USE FIXED SUBSET
# ------------------------------------------------------------
SUBSET_PATH = os.path.join(
    BASE_DIR,
    "fixed_subset_1000_with_text.csv"
)

df_subset = pd.read_csv(SUBSET_PATH)

print(f"Loaded subset: {len(df_subset)}")

# ============================================================
# CHECKPOINT RESUME
# ============================================================
if os.path.exists(CHECKPOINT):

    existing = pd.read_csv(CHECKPOINT)

    done_keys = set(
        zip(existing["id"], existing["prompt_type"])
    )

else:

    existing = pd.DataFrame()

    done_keys = set()

rows = []

# ============================================================
# EMOTION CLUSTERS
# ============================================================
CLUSTERS = [
    ["anger", "annoyance", "disapproval"],
    ["disgust", "disapproval"],
    ["fear", "nervousness"],
    [
        "joy",
        "amusement",
        "excitement",
        "pride",
        "optimism",
        "love",
        "admiration",
        "relief",
        "gratitude",
        "caring",
        "desire"
    ],
    ["confusion", "curiosity", "realization", "surprise"],
    ["sadness", "disappointment", "remorse", "grief"],
    ["surprise", "realization"],
]

emotion_to_cluster = {}

for cluster in CLUSTERS:

    for e in cluster:

        emotion_to_cluster[e] = cluster

# ============================================================
# HELPERS
# ============================================================
def sample_target_emotion(emotion_list):

    if isinstance(emotion_list, str):

        emotion_list = json.loads(emotion_list)

    return random.choice(emotion_list)

def is_neutral(labels):

    if isinstance(labels, str):

        labels = json.loads(labels)

    return len(labels) == 1 and labels[0] == "neutral"

def pick_distractor(
    target,
    gt_labels,
    mode="same_cluster"
):

    gt_set = set(gt_labels)

    all_emotions = list(emotion_to_cluster.keys())

    target_cluster = emotion_to_cluster.get(
        target,
        []
    )

    if mode == "same_cluster":

        candidates = [
            e for e in target_cluster
            if e != target and e not in gt_set
        ]

    else:

        candidates = [
            e for e in all_emotions
            if e not in target_cluster and e not in gt_set
        ]

    if not candidates:

        candidates = [
            e for e in all_emotions
            if e != target and e not in gt_set
        ]

    return random.choice(candidates)

# ============================================================
# PROMPT BUILDER
# ============================================================
def build_subtle_prompt(labels, mode=None):

    target = sample_target_emotion(labels)

    if mode is None:

        return (
            "You are a helpful assistant.",
            "What subtle emotion is expressed in this speech?",
            target,
            None,
            None
        )

    distractor = pick_distractor(
        target,
        labels,
        mode
    )

    while distractor == target or distractor in labels:

        distractor = pick_distractor(
            target,
            labels,
            mode
        )

    choices = [target, distractor, "neutral"]

    random.shuffle(choices)

    choice_str = "\n".join(
        [
            f"{chr(65+i)}. {c}"
            for i, c in enumerate(choices)
        ]
    )

    system_prompt = """
You are a speech emotion classifier.

Choose the most appropriate fine-grained emotion expressed in the speaker’s voice.

You must choose ONLY ONE option.

Return ONLY valid JSON:
{"label": "<one of the choices>", "confidence": <0.0-1.0>}
"""

    user_prompt = f"""
Which of the following best describes the speaker’s emotion?

Choices:
{choice_str}
"""

    return (
        system_prompt,
        user_prompt,
        target,
        distractor,
        choices
    )

# ============================================================
# RUN MODEL
# ============================================================

# ------------------------------------------------------------
# ONLY SAME_CLUSTER FOR APPENDIX
# ------------------------------------------------------------
PROMPT_TYPES = ["same_cluster"]

for idx, row in tqdm(
    df_subset.iterrows(),
    total=len(df_subset)
):

    wav_path = resolve_path(row["audio_path"])

    if not os.path.exists(wav_path):
        continue

    labels = json.loads(row["labels"])

    # --------------------------------------------------------
    # NEW: TRANSCRIPT
    # --------------------------------------------------------
    transcript = str(row["Utterance"]).strip()

    for prompt_type in PROMPT_TYPES:

        key = (idx, prompt_type)

        if key in done_keys:
            continue

        try:

            # =================================================
            # BUILD ORIGINAL PROMPT
            # =================================================
            system_prompt, user_prompt, target, distractor, choices = \
                build_subtle_prompt(
                    labels,
                    "same_cluster"
                )

            # =================================================
            # NEW: APPEND TRANSCRIPT
            # DO NOT MODIFY ORIGINAL PROMPT
            # =================================================
            multimodal_user_prompt = f"""
Transcript:
{transcript}

{user_prompt}
"""

            # =================================================
            # RUN MODEL
            # =================================================
            label, conf, raw, reps = judge_qwen2_audio(
                wav_path,
                system_prompt,
                multimodal_user_prompt,
                temperature=0.0
            )

            rep_dir = os.path.join(
                SAVE_DIR,
                "reps",
                prompt_type
            )

            os.makedirs(rep_dir, exist_ok=True)

            rep_path = os.path.join(
                rep_dir,
                f"{idx}.pt"
            )

            # optional
            # save_representation_tensor(reps, rep_path)

            rows.append({

                "id": idx,

                "audio_path": wav_path,

                "labels": row["labels"],

                "language": row["language"],

                # --------------------------------------------
                # NEW
                # --------------------------------------------
                "transcript": transcript,

                "prompt_type": "same_cluster_multimodal",

                "target": target,

                "pred": label,

                "confidence": conf,

                "raw": raw
            })

        except RuntimeError as e:

            if "out of memory" in str(e):

                print("OOM skip")

                cleanup()

                continue

            else:

                print(e)

                continue

        finally:

            cleanup()

    # ========================================================
    # CHECKPOINT SAVE
    # ========================================================
    if len(rows) >= 50:

        temp = pd.DataFrame(rows)

        combined = pd.concat(
            [existing, temp],
            ignore_index=True
        )

        combined.to_csv(CHECKPOINT, index=False)

        existing = combined

        rows = []

# ============================================================
# FINAL SAVE
# ============================================================

partial_df = pd.concat(
    [existing, pd.DataFrame(rows)],
    ignore_index=True
)

partial_df.to_csv(CHECKPOINT, index=False)

# ============================================================
# MERGE WITH DEFAULT RUN
# ============================================================

print("Merging with default results...")

default_df = pd.read_csv(DEFAULT_RESULTS_PATH)

# normalize paths
default_df["audio_path"] = default_df["audio_path"].apply(
    resolve_path
)

partial_df["audio_path"] = partial_df["audio_path"].apply(
    resolve_path
)

# filter default to subset
subset_paths = set(partial_df["audio_path"])

default_subset = default_df[
    default_df["audio_path"].isin(subset_paths)
].copy()

default_subset = default_subset.rename(columns={
    "pred_label": "default_pred",
    "confidence": "default_conf"
})

# ============================================================
# FINAL MERGE
# ============================================================

final_df = partial_df.merge(
    default_subset[["audio_path", "default_pred"]],
    on="audio_path",
    how="left"
)

final_df.to_csv(FINAL_OUTPUT, index=False)

print("Final saved:", FINAL_OUTPUT)